In [1]:
import pandas as pd
import numpy as np
from pandas.api.types import CategoricalDtype
DATA_PATH = r"C:\grad2_out\taxi_with_weather_FULL_dataset"  

needed_columns = [
    "PULocationID",
    "tpep_pickup_datetime",
    "temp_c",
    "rain_mm",
    "weather_code"
]

df = pd.read_parquet(DATA_PATH, columns=needed_columns, engine="pyarrow")
print(f"Total Raw Rows: {len(df)}")

Total Raw Rows: 85604543


In [2]:
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")


In [3]:
if df["tpep_pickup_datetime"].isna().any() or df["PULocationID"].isna().any():
    df.dropna(subset=["tpep_pickup_datetime", "PULocationID"], inplace=True)

In [4]:
df["time_bucket"] = df["tpep_pickup_datetime"].dt.floor("15min")
print(df["time_bucket"].dt.minute.value_counts().sort_index())

time_bucket
0     21398486
15    21204902
30    21387029
45    21614126
Name: count, dtype: int64


In [5]:
agg_pickups = (
    df.groupby(["PULocationID", "time_bucket"], as_index=False)
      .agg(pickup_cnt=("PULocationID", "size"))
)
print("Agg pickups shape:", agg_pickups.shape)
print("Zones:", agg_pickups["PULocationID"].nunique())



Agg pickups shape: (6258422, 3)
Zones: 263


In [6]:
weather = (
    df.groupby("time_bucket", as_index=False)
      .agg(
          temp_c=("temp_c", "mean"),
          rain_mm=("rain_mm", "mean"),
          weather_code=("weather_code", lambda s: s.mode().iat[0] if not s.mode().empty else s.iloc[0]),
      )
)

In [7]:
min_time = agg_pickups["time_bucket"].min()
max_time = agg_pickups["time_bucket"].max()
full_time = pd.date_range(start=min_time, end=max_time, freq="15min")

zones = agg_pickups["PULocationID"].unique()
full_index = pd.MultiIndex.from_product([zones, full_time], names=["PULocationID", "time_bucket"])

agg = (
    agg_pickups.set_index(["PULocationID", "time_bucket"])
              .reindex(full_index)
              .reset_index()
)


In [8]:
agg["pickup_cnt"] = agg["pickup_cnt"].fillna(0).astype("int32")


In [9]:
weather = (
    weather.set_index("time_bucket")
           .reindex(full_time)
           .ffill()
           .reset_index()
           .rename(columns={"index": "time_bucket"})
)

In [10]:
agg = agg.merge(weather, on="time_bucket", how="left")


In [11]:
agg = agg.dropna(subset=["weather_code"]).copy()

agg["is_rain"] = (agg["rain_mm"].fillna(0) > 0).astype("int8")
print("Final grid shape:", agg.shape)



Final grid shape: (17673600, 7)


In [12]:
agg["hour"] = agg["time_bucket"].dt.hour.astype("int16")
agg["minute"] = agg["time_bucket"].dt.minute.astype("int8")
agg["day_of_week"] = agg["time_bucket"].dt.dayofweek.astype("int8")
agg["is_weekend"] = (agg["day_of_week"] >= 5).astype("int8")
agg["month"] = agg["time_bucket"].dt.month.astype("int8")

In [13]:
BUCKET = pd.Timedelta(minutes=15)
HORIZON_BUCKETS = 1  # target = shift(-1)

max_tb = agg["time_bucket"].max()
last_feat_time = max_tb - (HORIZON_BUCKETS * BUCKET)
cutoff = last_feat_time - pd.Timedelta(days=28)


In [14]:
agg = agg.sort_values(["PULocationID", "time_bucket"]).reset_index(drop=True)

agg["pickup_cnt_raw"] = agg["pickup_cnt"].astype("int32")

q = 0.999
cap_by_zone = (
    agg.loc[agg["time_bucket"] < cutoff]
      .groupby("PULocationID")["pickup_cnt_raw"]
      .quantile(q)
)

agg = agg.join(cap_by_zone.rename("cap"), on="PULocationID")
agg["cap"] = agg["cap"].fillna(np.inf)  # zones with no train history => no cap

agg["pickup_cnt"] = np.minimum(agg["pickup_cnt_raw"], agg["cap"]).round().astype("int32")
agg["is_capped"] = (agg["pickup_cnt_raw"] > agg["pickup_cnt"]).astype("int8")
agg.drop(columns=["cap"], inplace=True)

In [15]:
import lightgbm as lgb


# ---------- 1) Feature engineering (lags/rolling/target) ----------
agg = agg.sort_values(["PULocationID", "time_bucket"]).reset_index(drop=True)
g = agg.groupby("PULocationID", sort=False)

agg["lag_1"]  = g["pickup_cnt"].shift(1)
agg["lag_4"]  = g["pickup_cnt"].shift(4)
agg["lag_96"] = g["pickup_cnt"].shift(96)

# rolling without leakage (use past only)
agg["pickup_shift1"] = g["pickup_cnt"].shift(1)

agg["roll_mean_1h"] = (
    agg.groupby("PULocationID")["pickup_shift1"]
       .rolling(4, min_periods=1)
       .mean()
       .reset_index(level=0, drop=True)
)

agg["roll_mean_3h"] = (
    agg.groupby("PULocationID")["pickup_shift1"]
       .rolling(12, min_periods=1)
       .mean()
       .reset_index(level=0, drop=True)
)

agg["target"] = g["pickup_cnt"].shift(-1)
agg.drop(columns=["pickup_shift1"], inplace=True)

C:\Users\A Store\AppData\Roaming\Python\Python312\site-packages\cupy\_environment.py:215: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


In [16]:
feature_cols = [
    "PULocationID",
    "pickup_cnt",
    "lag_1", "lag_4", "lag_96",
    "roll_mean_1h", "roll_mean_3h",
    "hour", "minute", "day_of_week", "is_weekend", "month",
    "temp_c", "rain_mm", "is_rain", "weather_code",
]

needed = feature_cols + ["target"]
train_df = agg.dropna(subset=needed).copy()

tr = train_df[train_df["time_bucket"] < cutoff].copy()
va = train_df[train_df["time_bucket"] >= cutoff].copy()

# categorical dtype ثابت من train -> valid
zone_dtype = CategoricalDtype(categories=sorted(tr["PULocationID"].unique()))
weather_dtype = CategoricalDtype(categories=sorted(tr["weather_code"].dropna().unique()))

for d in (tr, va):
    d["PULocationID"] = d["PULocationID"].astype(zone_dtype)
    d["weather_code"] = d["weather_code"].astype(weather_dtype)

X_train, y_train = tr[feature_cols], tr["target"]
X_valid, y_valid = va[feature_cols], va["target"]

print("Train rows:", len(tr), "Valid rows:", len(va))
print("X_train shape:", X_train.shape, "X_valid shape:", X_valid.shape)

Train rows: 16940882 Valid rows: 707207
X_train shape: (16940882, 16) X_valid shape: (707207, 16)


In [26]:
train_set = lgb.Dataset(
    X_train, label=y_train,
    categorical_feature=["PULocationID", "weather_code"],
    free_raw_data=False
)
valid_set = lgb.Dataset(
    X_valid, label=y_valid,
    categorical_feature=["PULocationID", "weather_code"],
    free_raw_data=False
)

params = {
    "objective": "poisson",
    "metric": ["poisson", "mae", "rmse"],
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "seed": 42,
    "verbose": -1,
}

model = lgb.train(
    params,
    train_set,
    num_boost_round=5000,
    valid_sets=[train_set, valid_set],
    valid_names=["train", "valid"],
    callbacks=[lgb.early_stopping(200), lgb.log_evaluation(100)]
)

Training until validation scores don't improve for 200 rounds
[100]	train's poisson: -11.3024	train's l1: 1.46858	train's rmse: 3.56264	valid's poisson: -12.8605	valid's l1: 1.59465	valid's rmse: 3.84695
[200]	train's poisson: -11.4931	train's l1: 1.22311	train's rmse: 3.31711	valid's poisson: -13.0194	valid's l1: 1.36516	valid's rmse: 3.5448
[300]	train's poisson: -11.5086	train's l1: 1.19183	train's rmse: 3.27288	valid's poisson: -13.0296	valid's l1: 1.33249	valid's rmse: 3.49629
[400]	train's poisson: -11.5141	train's l1: 1.17964	train's rmse: 3.23256	valid's poisson: -13.0334	valid's l1: 1.32012	valid's rmse: 3.4586
[500]	train's poisson: -11.5176	train's l1: 1.17261	train's rmse: 3.20935	valid's poisson: -13.0355	valid's l1: 1.31211	valid's rmse: 3.43723
[600]	train's poisson: -11.5203	train's l1: 1.16713	train's rmse: 3.19162	valid's poisson: -13.0372	valid's l1: 1.30631	valid's rmse: 3.4216
[700]	train's poisson: -11.5226	train's l1: 1.16233	train's rmse: 3.17614	valid's poisson

In [27]:
import os, pickle

HORIZON_LABEL = "tplus15m"  
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(MODEL_DIR, f"lgbm_demand_{HORIZON_LABEL}.txt")
bundle_path = os.path.join(MODEL_DIR, f"lgbm_bundle_{HORIZON_LABEL}.pkl")

# 1) Save the LightGBM booster
model.save_model(model_path)

# 2) Save metadata needed for correct inference
bundle = {
    "horizon_minutes": 15,
    "horizon_buckets": 1,
    "bucket_minutes": 15,
    "model_path": model_path,
    "feature_cols": feature_cols,
    "params": params,
    "zone_categories": list(zone_dtype.categories),
    "weather_categories": list(weather_dtype.categories),
}

with open(bundle_path, "wb") as f:
    pickle.dump(bundle, f)

print("Saved model:", model_path)
print("Saved bundle:", bundle_path)

Saved model: models\lgbm_demand_tplus15m.txt
Saved bundle: models\lgbm_bundle_tplus15m.pkl


In [29]:
import pickle
import lightgbm as lgb
import numpy as np
import pandas as pd
from pandas.api.types import CategoricalDtype

bundle_path = "models/lgbm_bundle_tplus15m.pkl"
with open(bundle_path, "rb") as f:
    bundle = pickle.load(f)

booster = lgb.Booster(model_file=bundle["model_path"])

feature_cols = bundle["feature_cols"]
zone_dtype = CategoricalDtype(categories=bundle["zone_categories"])
weather_dtype = CategoricalDtype(categories=bundle["weather_categories"])

print("Loaded model trees:", booster.num_trees())
print("Feature cols:", len(feature_cols))

Loaded model trees: 4998
Feature cols: 16


In [30]:
Xv = X_valid.copy()
Xv["PULocationID"] = Xv["PULocationID"].astype(zone_dtype)
Xv["weather_code"] = Xv["weather_code"].astype(weather_dtype)
Xv = Xv[feature_cols]

pred = booster.predict(Xv, num_iteration=booster.num_trees())

mae = np.mean(np.abs(pred - y_valid.to_numpy()))
rmse = np.sqrt(np.mean((pred - y_valid.to_numpy())**2))

print("MAE:", mae)
print("RMSE:", rmse)

baseline = Xv["lag_1"].to_numpy()
base_mae = np.mean(np.abs(baseline - y_valid.to_numpy()))
print("Baseline MAE (lag_1):", base_mae)

MAE: 1.2585474225840934
RMSE: 3.2787603269637153
Baseline MAE (lag_1): 1.7360419226619646


In [31]:
BUCKET = pd.Timedelta(minutes=bundle.get("bucket_minutes", 15))
H = bundle.get("horizon_buckets", 1)

max_tb = agg["time_bucket"].max()
feat_time = max_tb - (H * BUCKET)  # آخر وقت ينفع نستخدمه كـ features

# ناخد صف واحد لكل zone عند feat_time
latest = agg[agg["time_bucket"] == feat_time].copy()

# لازم يكون عنده كل features
latest = latest.dropna(subset=feature_cols)

# cast categories بنفس train
latest["PULocationID"] = latest["PULocationID"].astype(zone_dtype)
latest["weather_code"] = latest["weather_code"].astype(weather_dtype)

X_latest = latest[feature_cols]
yhat = booster.predict(X_latest, num_iteration=booster.num_trees())

out = latest[["PULocationID", "time_bucket"]].copy()
out["predict_time"] = out["time_bucket"] + BUCKET
out["yhat"] = yhat

# هات الـ actual من agg للـ predict_time (لو موجود)
actual = agg[["PULocationID", "time_bucket", "pickup_cnt"]].rename(
    columns={"time_bucket": "predict_time", "pickup_cnt": "y_true"}
)
out = out.merge(actual, on=["PULocationID", "predict_time"], how="left")

# رتب واعرض
out = out.sort_values(["PULocationID"]).reset_index(drop=True)
display(out.head(20))

# تقييم سريع على نفس الـ next-step (على zones اللي عندها y_true)
mask = out["y_true"].notna()
if mask.any():
    mae_next = np.mean(np.abs(out.loc[mask, "yhat"] - out.loc[mask, "y_true"]))
    rmse_next = np.sqrt(np.mean((out.loc[mask, "yhat"] - out.loc[mask, "y_true"])**2))
    print("Next-step MAE:", mae_next)
    print("Next-step RMSE:", rmse_next)
else:
    print("No y_true available for predict_time (maybe last bucket not present).")

,PULocationID,time_bucket,predict_time,yhat,y_true
0,1,2025-11-30 23:30:00,2025-11-30 23:45:00,0.082626,0
1,2,2025-11-30 23:30:00,2025-11-30 23:45:00,0.002321,0
2,3,2025-11-30 23:30:00,2025-11-30 23:45:00,0.033743,0
3,4,2025-11-30 23:30:00,2025-11-30 23:45:00,1.577978,1
4,5,2025-11-30 23:30:00,2025-11-30 23:45:00,0.000043,0
5,6,2025-11-30 23:30:00,2025-11-30 23:45:00,0.018354,0
6,7,2025-11-30 23:30:00,2025-11-30 23:45:00,2.385993,8
7,8,2025-11-30 23:30:00,2025-11-30 23:45:00,0.008842,0
8,9,2025-11-30 23:30:00,2025-11-30 23:45:00,0.025673,0
9,10,2025-11-30 23:30:00,2025-11-30 23:45:00,0.706707,1


Next-step MAE: 1.0094414810266616
Next-step RMSE: 2.5820316135718975


In [32]:
import numpy as np
import pandas as pd

BUCKET = pd.Timedelta(minutes=15)
H = 1

end_tb = agg["time_bucket"].max() - H * BUCKET
start_tb = end_tb - pd.Timedelta(days=7)

mask = (agg["time_bucket"] >= start_tb) & (agg["time_bucket"] <= end_tb)
window = agg.loc[mask].copy()

# نستخدم نفس feature_cols + نفس category mapping
window = window.dropna(subset=feature_cols)

window["PULocationID"] = window["PULocationID"].astype(zone_dtype)
window["weather_code"] = window["weather_code"].astype(weather_dtype)

Xw = window[feature_cols]
yhat = booster.predict(Xw, num_iteration=booster.num_trees())

pred_df = window[["PULocationID", "time_bucket"]].copy()
pred_df["predict_time"] = pred_df["time_bucket"] + BUCKET
pred_df["yhat"] = yhat

actual = agg[["PULocationID", "time_bucket", "pickup_cnt"]].rename(
    columns={"time_bucket": "predict_time", "pickup_cnt": "y_true"}
)

pred_df = pred_df.merge(actual, on=["PULocationID", "predict_time"], how="left").dropna(subset=["y_true"])

mae = np.mean(np.abs(pred_df["yhat"] - pred_df["y_true"]))
rmse = np.sqrt(np.mean((pred_df["yhat"] - pred_df["y_true"])**2))
print("7-day next-step MAE:", mae)
print("7-day next-step RMSE:", rmse)

7-day next-step MAE: 1.2227995053165435
7-day next-step RMSE: 3.139593111630704


In [34]:
out["yhat_int"] = np.clip(np.rint(out["yhat"]), 0, None).astype(int)
print(out["yhat_int"].describe())

count    263.000000
mean       2.437262
std       10.657169
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max      118.000000
Name: yhat_int, dtype: float64


In [36]:
import numpy as np
import lightgbm as lgb

sample = X_valid.sample(5000, random_state=42)

pred_mem = model.predict(sample)

booster = lgb.Booster(model_file=bundle["model_path"])
pred_file = booster.predict(sample)

diff = np.max(np.abs(pred_mem - pred_file))
print("Max abs diff between in-memory vs saved model:", diff)

Max abs diff between in-memory vs saved model: 0.0


In [37]:
import numpy as np

gold = X_valid.sample(1000, random_state=7).copy()
gold_pred = booster.predict(gold[feature_cols])

np.save("models/golden_pred_tplus15m.npy", gold_pred)
gold[feature_cols].to_parquet("models/golden_input_tplus15m.parquet", index=False)

print("Saved golden test input/preds")

Saved golden test input/preds


In [41]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from pandas.api.types import CategoricalDtype

# load bundle + model
with open("models/lgbm_bundle_tplus15m.pkl", "rb") as f:
    bundle = pickle.load(f)

booster_disk = lgb.Booster(model_file=bundle["model_path"])
feature_cols = bundle["feature_cols"]

# load golden files
gold_in = pd.read_parquet("models/golden_input_tplus15m.parquet")
gold_true = np.load("models/golden_pred_tplus15m.npy")

# ✅ enforce categorical mapping EXACTLY like training
zone_dtype = CategoricalDtype(categories=bundle["zone_categories"])
weather_dtype = CategoricalDtype(categories=bundle["weather_categories"])

gold_in["PULocationID"] = gold_in["PULocationID"].astype(zone_dtype)
gold_in["weather_code"] = gold_in["weather_code"].astype(weather_dtype)

# ensure same col order
gold_in = gold_in[feature_cols]

gold_api = booster_disk.predict(gold_in)

print("max abs diff:", np.max(np.abs(gold_api - gold_true)))
print("mean abs diff:", np.mean(np.abs(gold_api - gold_true)))

max abs diff: 0.0
mean abs diff: 0.0


In [28]:
dup = agg.duplicated(subset=["PULocationID", "time_bucket"]).sum()
print("Duplicates (zone,time_bucket):", dup)

if dup > 0:
    dups = agg[agg.duplicated(subset=["PULocationID", "time_bucket"], keep=False)] \
              .sort_values(["PULocationID", "time_bucket"])
    print("Sample duplicates:")
    display(dups.head(20))
else:
    print("✅ No duplicates: each (PULocationID, time_bucket) appears once.")

Duplicates (zone,time_bucket): 0
✅ No duplicates: each (PULocationID, time_bucket) appears once.


In [18]:
dup = agg.duplicated(subset=["PULocationID", "time_bucket"]).sum()
print("Duplicates (zone,time_bucket):", dup)

Duplicates (zone,time_bucket): 0


In [19]:
step_ok = (
    agg.groupby("PULocationID")["time_bucket"]
       .diff()
       .dropna()
       .eq(pd.Timedelta(minutes=15))
       .mean()
)
print("Fraction of 15-min steps:", step_ok)

Fraction of 15-min steps: 1.0


In [22]:
cols = [
    "PULocationID",
    "pickup_cnt",
    "lag_1", "lag_4", "lag_96",
    "roll_mean_1h", "roll_mean_3h",
    "hour", "minute", "day_of_week", "is_weekend", "month",
    "temp_c", "rain_mm", "is_rain", "weather_code",
]

# 1) عرض 10 صفوف عشوائي (أوضح من head)
display(train_df[cols].sample(10, random_state=42))

# 2) عرض أول 20 صف (لو تحب تشوف التسلسل)
display(train_df[cols].head(20))

# 3) ملخص سريع لكل عمود: dtype + nulls + unique + min/max
summary = pd.DataFrame({
    "dtype": train_df[cols].dtypes.astype(str),
    "nulls": train_df[cols].isna().sum(),
    "null_%": (train_df[cols].isna().mean() * 100).round(3),
    "unique": train_df[cols].nunique(),
})

# min/max للأعمدة الرقمية فقط
num_cols = train_df[cols].select_dtypes(include="number").columns
summary.loc[num_cols, "min"] = train_df[num_cols].min().values
summary.loc[num_cols, "max"] = train_df[num_cols].max().values

display(summary.sort_values("nulls", ascending=False))

,PULocationID,pickup_cnt,lag_1,lag_4,lag_96,roll_mean_1h,roll_mean_3h,hour,minute,day_of_week,is_weekend,month,temp_c,rain_mm,is_rain,weather_code
7947592,121,0,0.0,1.0,0.0,0.25,0.166667,10,0,5,1,7,29.837760,0.000000,0,3
15372938,231,50,51.0,45.0,44.0,46.25,35.166667,18,30,2,0,6,22.841033,2.402299,1,61
8373336,127,0,1.0,0.0,0.0,0.25,0.083333,6,0,2,0,2,2.346882,0.000000,0,0
11756495,177,3,1.0,2.0,0.0,1.50,1.166667,11,45,5,1,10,13.278128,0.000000,0,0
11743009,177,0,1.0,1.0,0.0,1.25,1.333333,0,15,5,1,6,23.048217,0.000000,0,3
15807931,238,13,8.0,17.0,9.0,14.00,19.500000,22,45,4,0,6,20.440874,0.000000,0,3
16709539,251,0,0.0,0.0,0.0,0.00,0.000000,16,45,2,0,4,5.976952,0.000000,0,3
15847850,238,33,22.0,26.0,34.0,23.75,25.500000,18,30,0,0,8,27.814954,0.000000,0,0
11455859,173,0,0.0,0.0,0.0,0.75,0.250000,20,45,2,0,11,5.449676,0.000000,0,3
15812447,238,2,12.0,8.0,2.0,9.00,12.750000,23,45,2,0,7,24.229792,0.000000,0,0


,PULocationID,pickup_cnt,lag_1,lag_4,lag_96,roll_mean_1h,roll_mean_3h,hour,minute,day_of_week,is_weekend,month,temp_c,rain_mm,is_rain,weather_code
96,1,0,0.0,0.0,0.0,0.0,0.0,0,0,1,0,1,-1.222740,0.0,0,0
97,1,0,0.0,0.0,0.0,0.0,0.0,0,15,1,0,1,-1.249589,0.0,0,0
98,1,0,0.0,0.0,0.0,0.0,0.0,0,30,1,0,1,-1.174376,0.0,0,0
99,1,0,0.0,0.0,0.0,0.0,0.0,0,45,1,0,1,-1.181273,0.0,0,0
100,1,0,0.0,0.0,0.0,0.0,0.0,1,0,1,0,1,-1.801960,0.0,0,0
101,1,0,0.0,0.0,0.0,0.0,0.0,1,15,1,0,1,-1.839121,0.0,0,0
102,1,0,0.0,0.0,0.0,0.0,0.0,1,30,1,0,1,-1.792079,0.0,0,0
103,1,0,0.0,0.0,0.0,0.0,0.0,1,45,1,0,1,-1.771429,0.0,0,0
104,1,0,0.0,0.0,0.0,0.0,0.0,2,0,1,0,1,-2.255015,0.0,0,0
105,1,0,0.0,0.0,0.0,0.0,0.0,2,15,1,0,1,-2.233154,0.0,0,0


,dtype,nulls,null_%,unique,min,max
PULocationID,int32,0,0.0,263,1.0,265.0
pickup_cnt,int32,0,0.0,239,0.0,238.0
lag_1,float64,0,0.0,239,0.0,238.0
lag_4,float64,0,0.0,239,0.0,238.0
lag_96,float64,0,0.0,239,0.0,238.0
roll_mean_1h,float64,0,0.0,928,0.0,238.0
roll_mean_3h,float64,0,0.0,2494,0.0,227.583333
hour,int16,0,0.0,24,0.0,23.0
minute,int8,0,0.0,4,0.0,45.0
day_of_week,int8,0,0.0,7,0.0,6.0


In [25]:
cols_dbg = ["PULocationID", "time_bucket", "pickup_cnt", "lag_1", "lag_4", "lag_96", "roll_mean_1h", "roll_mean_3h"]
display(train_df.loc[train_df["pickup_cnt"].eq(4), cols_dbg].head(20))

,PULocationID,time_bucket,pickup_cnt,lag_1,lag_4,lag_96,roll_mean_1h,roll_mean_3h
1698,1,2024-01-18 16:30:00,4,0.0,0.0,1.0,0.00,0.166667
2951,1,2024-01-31 17:45:00,4,0.0,1.0,0.0,0.25,0.250000
3317,1,2024-02-04 13:15:00,4,0.0,0.0,0.0,0.00,0.333333
3331,1,2024-02-04 16:45:00,4,0.0,0.0,0.0,0.00,0.166667
3821,1,2024-02-09 19:15:00,4,0.0,0.0,0.0,0.00,0.166667
3990,1,2024-02-11 13:30:00,4,0.0,0.0,0.0,0.00,0.083333
4003,1,2024-02-11 16:45:00,4,0.0,0.0,2.0,0.75,0.333333
4672,1,2024-02-18 16:00:00,4,3.0,1.0,0.0,1.50,0.750000
4768,1,2024-02-19 16:00:00,4,0.0,0.0,4.0,0.50,0.416667
6510,1,2024-03-08 19:30:00,4,3.0,0.0,0.0,0.75,0.250000


In [24]:
z = 1
tmp = agg[agg["PULocationID"] == z]

print("rows:", len(tmp))
print("non-zero rows:", (tmp["pickup_cnt"] > 0).sum())
print("zero %:", (tmp["pickup_cnt"] == 0).mean())
print("max pickup_cnt:", tmp["pickup_cnt"].max())

rows: 67200
non-zero rows: 8682
zero %: 0.8708035714285715
max pickup_cnt: 4
